## Fetching Actors from TMDB and Creating Database Scripts

In this Jupyter notebook, I will fetch detailed actor information from TMDB for each movie in my dataset. After retrieving the data, I will generate SQL scripts to insert the actors and their movie associations into the database.

In [ ]:
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import requests
from dotenv import load_dotenv

from concurrent.futures import ThreadPoolExecutor, as_completed

# Load the environment variables from .env
load_dotenv()

Create the functions to fetch the data from TMDB

In [ ]:
def fetch_tmdb_data(movie_id, api_key):
    """Generic function to fetch any TMDB data for a movie"""
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={api_key}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:  # Too Many Requests
            # Sleep for a bit if we hit rate limits
            time.sleep(2)
            return fetch_tmdb_data(movie_id, api_key)  # Simple retry
        else:
            print(f"Error: HTTP {response.status_code} for movie {movie_id}")
            return None
    except Exception as e:
        print(f"Error fetching data for movie {movie_id}: {e}")
        return None

In [ ]:
def fetch_movies(ids, api_key, max_workers=10, delay=0.05):
    """
    Function to fetch movie data for multiple movie IDs concurrently
    
    Args:
        ids: List of movie IDs
        api_key: TMDB API key
        max_workers: Max number of concurrent requests
        delay: Small delay between requests to be nice to API
        
    Returns:
        (successful_results, failed_ids)
    """
    successful = []
    failed = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_tmdb_data, movie_id, api_key): movie_id for movie_id in ids}
        
        for future in as_completed(futures):
            movie_id = futures[future]
            result = future.result()
            
            if result:
                successful.append(result)
            else:
                failed.append(movie_id)
            
            # Small delay to avoid hammering the API
            time.sleep(delay)
    
    return successful, failed

Read the id of the movies I selected for my final dataset 

In [ ]:
ids = pd.read_csv('../data/processed/clean_movies_ids.csv')
ids.head()

In [ ]:
api_key = os.getenv('tmdb_api_key')
movie_ids = ids['id']

Fetch the credits 

In [ ]:
# Fetching credits
credits, failed_credits = fetch_movies(movie_ids, api_key)
print(f"Fetched {len(credits)} movie credits, failed {len(failed_credits)}")

In [ ]:
credits = pd.DataFrame(credits)
credits.head()

### Preprocessing

Now that I have the data, I need to convert it to the correct data types. First, I will ignore the crew information, and then I will select the first five actors, as they are typically the most important for each movie.

In [ ]:
# Process cast data: parse JSON, keep first 5 members with selected details, and rename movie ID
df = (
    credits[['id', 'cast']]
    .assign(
        cast=lambda d: d['cast'].apply(
            lambda x: [
                {"id": item["id"], "name": item["name"], "profile_path": item["profile_path"]}
                for item in (json.loads(x) if isinstance(x, str) else x)[:5]
            ]
        )
    )
    .rename(columns={'id': 'movie_id'})
)

df.head()

In [ ]:
# Check the structure of the cast data
df.iloc[0]['cast']

Now I need to extract all the details from the actors dictionary (id, name, and profile path) and explode the list so that each row represents a single movie–actor combination. This way, every actor in a movie will have its own row in the dataset.

In [ ]:
# Expand the cast list so each actor gets its own row
actors_df = df.explode('cast')

# Cast column contains dictionaries; split each dictionary into separate columns
actors_df = pd.concat(
    [
        actors_df.drop(columns=['cast']),
        actors_df['cast'].apply(lambda x: pd.Series(x) if isinstance(x, dict) else pd.Series())  # Extract actor fields
    ],
    axis=1
)

actors_df.head()

Check how many nulls values we have

In [ ]:
actors_df.isna().sum()

Some movies like the animated does not have any cast information. There are 16 movies without cast information and 904 actors without a profile path image. I will drop the movies that do not have information about their actors

In [ ]:
# Drop rows where actor ID is missing, and convert ID to integer
actors_df = actors_df.dropna(subset=['id'])
actors_df['id'] = actors_df['id'].astype(int)
actors_df.head()

 Now I have to create 2 separate dataframes, the movie-actors and the actors to then create the sql script and insert them into the database

In [ ]:
# Create a unique table of actors with their profile paths
actors_table = actors_df[['id', 'name', 'profile_path']].drop_duplicates(subset='id').reset_index(drop=True)
actors_table.head()

In [ ]:
# Keep only the TMDB ID and actor id
movie_actor_table = actors_df[['movie_id', 'id']].rename(columns={'id': 'actor_id'}).drop_duplicates()
movie_actor_table.head()

In [ ]:
# Since the database has an internal actor id, I will recreate the id for the actors and maintain consistency
actors_table['internal_id'] = range(1, len(actors_table) + 1)
actors_table.head()

In [ ]:
# Merge movie_actor_table with actors_table to get the internal actor IDs
movie_actor_table = movie_actor_table.merge(
    actors_table[['id', 'internal_id']],
    left_on='actor_id',
    right_on='id',
    how='left'
).drop(columns=['id', 'actor_id']) \
 .rename(columns={'internal_id': 'actor_id'})

# Preview the result
movie_actor_table.head()

See how many actors we have and how many movie-actors

In [ ]:
print(f"There are {actors_table.shape[0]} unique actors and {movie_actor_table.shape[0]} movie-actor relationships.")

# Merging the actors with the internal movie id

Since in my database I am using another id for movies (internal movie id) I have to merge the movies with my dataframe to have consistency across the database

In [ ]:
movies_df = pd.read_csv('../data/processed/movies_final_spanish.csv')
movies_df.head().transpose()

In [ ]:
# Keep only the movie ID and TMDB ID, and rename TMDB for merging
movies_df = movies_df[['id', 'TMDB_id']].rename(columns={'TMDB_id': 'movie_id'})
# Add one since our movie IDs start from 1, but the original CSV starts from 0
movies_df['id'] = movies_df['id'] + 1
movies_df.head()

In [ ]:
# Merge to get the final movie-actor relationships with our internal movie IDs
movie_actor = movies_df.merge(movie_actor_table, on='movie_id', how='inner')[['id', 'actor_id']].rename(columns={'id': 'movie_id'})
movie_actor.head()

### SQL Script

Now I will generate SQL scripts to insert the processed data into the database.

In [ ]:
# ACTORS SQL FILE
with open('../../database/seed_actors.sql', 'w', encoding='utf-8') as f:
    f.write("INSERT INTO actors (tmdb_id, name, profile_path) VALUES\n")

    actor_values = []
    for _, row in actors_table.iterrows():
        tmdb_id = row['id']
        name = row['name'].replace("'", "''")  # Escape single quotes
        profile_path = row['profile_path'] if pd.notna(row['profile_path']) else 'NULL'
        profile_path = f"'{profile_path}'" if profile_path != 'NULL' else 'NULL'
        actor_values.append(f"({tmdb_id}, '{name}', {profile_path})")

    f.write(",\n".join(actor_values) + ";\n")

In [ ]:
# MOVIE_ACTORS SQL FILE
with open('../../database/seed_movie_actors.sql', 'w', encoding='utf-8') as f:
    f.write("INSERT INTO movie_actors (movie_id, actor_id) VALUES\n")

    movie_actor_values = []
    for _, row in movie_actor.iterrows():
        movie_actor_values.append(f"({row['movie_id']}, {row['actor_id']})")

    f.write(",\n".join(movie_actor_values) + ";\n")